# L2CS-Net Knowledge Distillation (ResNet50 → MobileNetV3-Small)

**目的**: L2CS-Net (ResNet50, ~25M params) の知識を MobileNetV3-Small (~2.5M params) に蒸留し、ブラウザ上でのリアルタイム視線推定を実現する。

**環境**: Google Colab (T4 GPU)
**実験管理**: Weights & Biases

## Setup

In [ ]:
# 1. GPU確認 & リポジトリクローン
!nvidia-smi
!git clone https://github.com/misanga1225/gdgoc.git /content/gdgoc
%cd /content/gdgoc/training

In [ ]:
# 2. 依存インストール
!pip install -q torch torchvision timm wandb onnx onnxruntime omegaconf gdown scipy tqdm opencv-python-headless

In [ ]:
# 3. W&B ログイン
import wandb
wandb.login()

## データセットの準備

Gaze360 データセットをダウンロードして展開します。  
初回実行時のみ必要です（Google Drive にキャッシュ推奨）。

In [ ]:
# 4. Google Drive マウント (チェックポイント・データ永続化)
from google.colab import drive
drive.mount('/content/drive')

# データディレクトリ (Drive上にキャッシュ)
DATA_DIR = "/content/drive/MyDrive/l2cs-distill/data"
CKPT_DIR = "/content/drive/MyDrive/l2cs-distill/checkpoints"

!mkdir -p {DATA_DIR} {CKPT_DIR}

In [ ]:
# 5. Gaze360 データセットのダウンロード
# 公式サイト: http://gaze360.csail.mit.edu/
# ※ ダウンロードURLはアカデミックライセンス同意後に取得してください
import os

if not os.path.exists(f"{DATA_DIR}/gaze360/metadata.mat"):
    print("Gaze360 データが見つかりません。")
    print("以下の手順でダウンロードしてください:")
    print("  1. http://gaze360.csail.mit.edu/ にアクセス")
    print("  2. ライセンスに同意してダウンロードリンクを取得")
    print("  3. 以下のセルのURLを書き換えて実行")
    print()
    print("または Google Drive に手動でアップロード:")
    print(f"  {DATA_DIR}/gaze360/")
else:
    print(f"Gaze360 データ検出: {DATA_DIR}/gaze360/")

In [ ]:
# 6. L2CS-Net 公式事前学習済み重み (Teacher) のダウンロード
import gdown

TEACHER_WEIGHTS = f"{DATA_DIR}/L2CSNet_gaze360.pkl"
if not os.path.exists(TEACHER_WEIGHTS):
    # 公式リポジトリの Gaze360 学習済みモデル
    # https://github.com/Ahmednull/L2CS-Net#pre-trained-models
    gdown.download(
        "https://drive.google.com/uc?id=1EnJEhKLuPgJOP3JpOia1QdDJDkfjwF4l",
        TEACHER_WEIGHTS,
        quiet=False,
    )
    print(f"Teacher weights: {TEACHER_WEIGHTS}")
else:
    print(f"Teacher weights already exist: {TEACHER_WEIGHTS}")

## 蒸留学習

config を Google Drive のパスに合わせて上書きし、学習を実行します。

In [ ]:
# 7. config を Colab 環境用にオーバーライドして書き出し
from omegaconf import OmegaConf

cfg = OmegaConf.load("configs/distill.yaml")

# Colab パスに合わせる
cfg.data.data_root = DATA_DIR
cfg.teacher.weights = "L2CSNet_gaze360.pkl"

# 必要に応じて調整
# cfg.training.batch_size = 64   # VRAM不足時
# cfg.training.epochs = 30
# cfg.wandb.entity = "your-wandb-entity"

colab_cfg_path = "configs/distill_colab.yaml"
OmegaConf.save(cfg, colab_cfg_path)
print(OmegaConf.to_yaml(cfg))

In [ ]:
# 8. 蒸留学習の実行
!python -m src.train --config configs/distill_colab.yaml

In [ ]:
# 9. チェックポイントを Google Drive にコピー
import shutil
shutil.copy("checkpoints/best_student.pth", f"{CKPT_DIR}/best_student.pth")
shutil.copy("checkpoints/final_student.pth", f"{CKPT_DIR}/final_student.pth")
print(f"Checkpoints saved to {CKPT_DIR}")

## ONNX変換

学習済みの Student モデルを ONNX (float16) に変換し、ブラウザデプロイ用に出力します。

In [ ]:
# 10. ONNX 変換 + float16 量子化
!python -m src.export_onnx \
    --checkpoint checkpoints/best_student.pth \
    --output outputs/l2cs_lite.onnx \
    --quantize fp16

In [ ]:
# 11. モデルサイズ確認 & Google Drive にコピー
import os

for name in ["l2cs_lite.onnx", "l2cs_lite_fp16.onnx"]:
    path = f"outputs/{name}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{name}: {size_mb:.1f} MB")
        shutil.copy(path, f"{CKPT_DIR}/{name}")

print(f"\nONNX models saved to {CKPT_DIR}")
print("Google Drive からローカルにダウンロードし、patient/ で使用してください。")